# 04 · Build recommender (reacting to the enemy's lineup and builds)
**Question:** for my hero, at this moment, given my items, my allies, the enemy heroes **and what the enemy team has bought so far**, which next item gives the best chance to win?

**Data:** `fe.purchase_events` gives one row per shop purchase (tier 2+), with `own_items`, `ally_heroes`, `enemy_heroes`, `enemy_items` (held at that second), enemy spend by category, the net worth gap and the result.

**The trap:** "players who bought X won more" is mostly "players who were already winning could afford X". Two guards:
1. condition on the net worth gap and the minute, so items are compared between players in the same situation;
2. compare items **against each other for the same hero at the same moment**, not against "not buying".

**Plan, simplest first:**
- **v0 lift table:** per hero, per item: win rate when bought in a time window vs the hero's average in that window, adjusted for net worth gap. Exports as `table@1`.
- **v1 contextual model:** per hero (or hero family), logistic regression on `win ~ item_bought × context`, where context = enemy hero multi-hot + enemy items multi-hot + enemy spend + nw gap + minute. At serve time, score each candidate item in the current context and rank by predicted win chance. Exports as `linear@1` with the interaction columns named `item_<id>__e_<id>` etc.
- **v2 (stretch):** "what do strong players buy here" as a next-item classifier on high-rank games, blended with v1.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from broadway_ml.flatten import load_tables, item_catalog
from broadway_ml import features as fe, evaluate as ev, export as ex
T = load_tables('../data/tables')
{k: len(v) for k, v in T.items()}

In [ ]:
cat = item_catalog()
p = T['players']
HERO = int(p.hero_id.value_counts().index[0])  # most played hero; change me
evts = fe.purchase_events(T['matches'], p, T['items'], T['stats'], cat, hero_ids=[HERO], min_tier=2)
print(HERO, len(evts), 'purchases'); evts.head()

In [ ]:
# What the enemy has built by the time this hero buys, e.g. how often they face heavy spirit damage
evts[['enemy_weapon_spend', 'enemy_vitality_spend', 'enemy_spirit_spend']].describe()

## v0: lift table (baseline)

In [ ]:
evts['minute_bin'] = pd.cut(evts.t / 60, [0, 10, 20, 30, 99], labels=['0-10', '10-20', '20-30', '30+'])
evts['ahead'] = evts.nw_diff > 0
base = evts.groupby(['minute_bin', 'ahead'], observed=True).win.mean().rename('base')
lift = (evts.join(base, on=['minute_bin', 'ahead'])
            .assign(delta=lambda d: d.win - d.base)
            .groupby(['item_id', 'minute_bin'], observed=True).agg(n=('win', 'size'), delta=('delta', 'mean')).reset_index())
lift = lift[lift.n >= 100].merge(cat[['item_id', 'name']], on='item_id')
lift.sort_values('delta', ascending=False).head(15)

## v1: your contextual model
Build the design matrix with `fe.multi_hot` for `enemy_heroes`, `enemy_items` and the candidate `item_id`, then interactions between the candidate item and the context. Evaluate with a **time split** and these checks:
- log loss / AUC vs the lift table, on held-out matches
- **ranking check:** for held-out purchases, does the model rank the item actually bought by eventual winners higher than the one bought by losers in similar spots?
- **sanity reads:** does it suggest anti-heal more against healing-heavy enemy builds, and bullet resist against weapon-heavy ones? (These are the stories for the CV.)

In [ ]:
# TODO(V): v1 model


In [ ]:
# Export v0 now so the site can show something while v1 is built
rows = {f"{HERO}:{r.item_id}:{r.minute_bin}": {"n": int(r.n), "delta": round(float(r.delta), 4)} for r in lift.itertuples()}
ex.export_table(rows, ['hero_id', 'item_id', 'minute_bin'], f'../models/build_lift_{HERO}.json', name='build-lift-v0',
                notes='win-rate change vs same-minute same-lead baseline; needs n >= 100')